In [ ]:
from __future__ import annotations

import os
import warnings
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    auc,
    brier_score_loss,
)

warnings.filterwarnings("ignore")

data_fs_static_external = pd.read_csv("YOUR_PATH")
data_fs_static_train = pd.read_csv("YOUR_PATH")

# Patient identifier used for grouped nested cross-validation and clustered bootstrap CIs.
# IMPORTANT: do not include this identifier in feature_space.
GROUP_COL = "subject_reference"

scale = 'yes'
feature_space = ['feature_1', 'feature_2', ...]

X_train, y_train  = do_train_test_split(data_fs_static_train,feature_space,scale)


In [ ]:
OUTER_SPLITS = 5
INNER_SPLITS_THRESHOLD = 5          # OOF folds used on outer-train to pick thresholds
N_NESTED_TRIALS = 20
RANDOM_SEEDS = [1000 + i for i in range(N_NESTED_TRIALS)]

N_BOOTSTRAPS = 2000
BOOT_ALPHA = 0.05

RUN_SHAP = True
MAX_SHAP_SAMPLES_PER_FOLD = 1000     # set None for all (can be large)
SHAP_BACKGROUND_N = 200              # background samples for LinearExplainer

OUT_DIR = "outputs_nested_glm"
FIG_DIR = os.path.join(OUT_DIR, "figures")       # kept for symmetry; no plots by default
SHAP_DIR = os.path.join(OUT_DIR, "shap_values")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(SHAP_DIR, exist_ok=True)

print("Ready.")


# ============================
# Helper functions (from your external script, adapted)
# ============================

def as_numpy(y: Iterable) -> np.ndarray:
    return np.asarray(y).reshape(-1)

def safe_confusion(yt: np.ndarray, yp: np.ndarray) -> Tuple[int, int, int, int]:
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return int(tn), int(fp), int(fn), int(tp)

def stable_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))

def stable_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    return float(auc(rec, prec))

def point_metrics(y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    tn, fp, fn, tp = safe_confusion(y_true, y_pred)
    spec = tn / (tn + fp + 1e-12)
    sens = tp / (tp + fn + 1e-12)
    npv  = tn / (tn + fn + 1e-12)

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(spec),
        "sensitivity": float(sens),
        "npv": float(npv),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "auc": float(stable_roc_auc(y_true, y_prob)),
        "auprc": float(stable_auprc(y_true, y_prob)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "tn": float(tn), "fp": float(fp), "fn": float(fn), "tp": float(tp),
    }

def thresholds_from_predictions(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)

    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    if len(thresh) == 0:
        return {"F1-optimal": 0.5, "MCC-optimal": 0.5, "Youden": 0.5}

    # F1-optimal
    f1_vals = 2 * (prec * rec) / (prec + rec + 1e-12)
    t_f1 = float(thresh[int(np.nanargmax(f1_vals[:-1]))])

    # MCC-optimal
    mcc_vals = [matthews_corrcoef(y_true, (y_prob >= t).astype(int)) for t in thresh]
    t_mcc = float(thresh[int(np.nanargmax(mcc_vals))])

    # Youden
    youden_vals = []
    for t in thresh:
        yp = (y_prob >= t).astype(int)
        tn, fp, fn, tp = safe_confusion(y_true, yp)
        sens = tp / (tp + fn + 1e-12)
        spec = tn / (tn + fp + 1e-12)
        youden_vals.append(sens + spec - 1)
    t_youden = float(thresh[int(np.nanargmax(youden_vals))])

    return {"F1-optimal": t_f1, "MCC-optimal": t_mcc, "Youden": t_youden}

def make_glm_pipeline(random_state: int) -> Pipeline:
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(
                penalty="none",
                solver="lbfgs",
                class_weight="balanced",
                max_iter=5000,
                random_state=random_state,
            )),
        ]
    )

def get_oof_probabilities(
    model: Pipeline,
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
    n_splits: int,
    seed: int,
) -> np.ndarray:
    """OOF probabilities with all admissions from each patient kept in one fold."""
    y = as_numpy(y)
    groups = as_numpy(groups)
    if not (len(X) == len(y) == len(groups)):
        raise ValueError("X, y, and groups must have identical lengths for OOF prediction.")
    if pd.isna(groups).any():
        raise ValueError(f"{GROUP_COL} contains missing values.")

    oof = np.full(shape=(len(X),), fill_value=np.nan, dtype=float)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for train_idx, valid_idx in cv.split(X, y, groups=groups):
        overlap = set(groups[train_idx]).intersection(set(groups[valid_idx]))
        if overlap:
            raise RuntimeError(
                f"Patient leakage detected in threshold-selection OOF CV: "
                f"{len(overlap)} overlapping patient(s)."
            )

        m = clone(model)
        m.fit(X.iloc[train_idx], y[train_idx])
        oof[valid_idx] = m.predict_proba(X.iloc[valid_idx])[:, 1]

    if np.isnan(oof).any():
        raise RuntimeError("OOF probabilities contain NaNs; check grouped CV/data.")
    return oof

def _logit(p: np.ndarray) -> np.ndarray:
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)

def calibration_in_the_large(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty="none", solver="lbfgs", max_iter=2000)
    lr.fit(z, as_numpy(y_true))
    return float(lr.intercept_[0])

def calibration_slope(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty="none", solver="lbfgs", max_iter=2000)
    lr.fit(z, as_numpy(y_true))
    return float(lr.coef_[0][0])

# ============================
# Cluster bootstrap (patient-level resampling)
# ============================

def _prepare_cluster_indices(groups: np.ndarray) -> Tuple[np.ndarray, List[np.ndarray]]:
    """Precompute admission-row indices belonging to each patient."""
    groups = as_numpy(groups)
    if pd.isna(groups).any():
        raise ValueError("Patient identifiers contain missing values; grouped resampling requires complete IDs.")

    unique_groups = pd.unique(groups)
    rows_by_group = [np.flatnonzero(groups == g) for g in unique_groups]
    return np.asarray(unique_groups, dtype=object), rows_by_group

def cluster_bootstrap_indices(
    rng: np.random.Generator,
    rows_by_group: List[np.ndarray],
) -> np.ndarray:
    """Resample patients with replacement, retaining all admissions for sampled patients."""
    n_groups = len(rows_by_group)
    sampled_group_positions = rng.integers(0, n_groups, size=n_groups, endpoint=False)
    return np.concatenate([rows_by_group[j] for j in sampled_group_positions])

def bootstrap_metric_distribution(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    y_pred: np.ndarray,
    groups: np.ndarray,
    metrics: List[str],
    n_boot: int,
    seed: int,
) -> pd.DataFrame:
    """Admission-level metrics with patient-clustered bootstrap confidence intervals."""
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    y_pred = as_numpy(y_pred)
    groups = as_numpy(groups)

    if not (len(y_true) == len(y_prob) == len(y_pred) == len(groups)):
        raise ValueError("y_true, y_prob, y_pred, and groups must have identical lengths.")

    rng = np.random.default_rng(seed)
    _, rows_by_group = _prepare_cluster_indices(groups)
    rows = []

    for b in range(n_boot):
        idx = cluster_bootstrap_indices(rng, rows_by_group)
        m = point_metrics(y_true[idx], y_prob[idx], y_pred[idx])
        for k in metrics:
            rows.append({"boot_id": b, "metric": k, "boot_value": float(m[k])})
    return pd.DataFrame(rows)

def bootstrap_calibration_distribution(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: np.ndarray,
    n_boot: int,
    seed: int,
) -> pd.DataFrame:
    """Patient-cluster bootstrap for Brier score, CITL, and calibration slope."""
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    groups = as_numpy(groups)

    if not (len(y_true) == len(y_prob) == len(groups)):
        raise ValueError("y_true, y_prob, and groups must have identical lengths.")

    rng = np.random.default_rng(seed)
    _, rows_by_group = _prepare_cluster_indices(groups)
    rows = []

    for b in range(n_boot):
        idx = cluster_bootstrap_indices(rng, rows_by_group)
        yt = y_true[idx]
        pr = y_prob[idx]

        rows.append({"boot_id": b, "metric": "brier",
                     "boot_value": float(brier_score_loss(yt, pr))})

        if len(np.unique(yt)) < 2:
            citl = np.nan
            slope = np.nan
        else:
            try:
                citl = float(calibration_in_the_large(yt, pr))
                slope = float(calibration_slope(yt, pr))
            except Exception:
                citl = np.nan
                slope = np.nan

        rows.append({"boot_id": b, "metric": "citl", "boot_value": citl})
        rows.append({"boot_id": b, "metric": "slope", "boot_value": slope})

    return pd.DataFrame(rows)

def _subsample_idx(n: int, max_n: Optional[int], seed: int) -> np.ndarray:
    if max_n is None or n <= max_n:
        return np.arange(n)
    rng = np.random.default_rng(seed)
    return rng.choice(n, size=max_n, replace=False)


# ============================
# SHAP (LinearExplainer on scaled space)
# ============================

def shap_for_glm_pipeline(
    fitted_model: Pipeline,
    X_train_fold: pd.DataFrame,     # background source (outer-train fold)
    X_test_fold: pd.DataFrame,      # explain on (outer-test fold)
    feature_names: List[str],
    max_shap_samples: Optional[int],
    background_n: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns:
      shap_values: (n_explain, n_features)
      idx_labels:  original index labels for explained rows
    """
    import shap  # must be installed

    scaler: StandardScaler = fitted_model.named_steps["scaler"]
    lr: LogisticRegression = fitted_model.named_steps["lr"]

    X_train_scaled = pd.DataFrame(
        scaler.transform(X_train_fold),
        columns=feature_names,
        index=X_train_fold.index,
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test_fold),
        columns=feature_names,
        index=X_test_fold.index,
    )

    # subsample test for explain
    test_idx = _subsample_idx(len(X_test_scaled), max_shap_samples, seed=seed)
    X_explain = X_test_scaled.iloc[test_idx]
    idx_labels = X_explain.index.to_numpy()

    # background sample
    bg_n = min(background_n, len(X_train_scaled))
    background = shap.sample(X_train_scaled, bg_n, random_state=seed)

    explainer = shap.LinearExplainer(lr, background)
    sv = np.asarray(explainer.shap_values(X_explain))
    return sv, idx_labels


# ============================
# Nested CV runner
# ============================

@dataclass
class NestedCVResult:
    seed: int
    y_true: np.ndarray
    p: np.ndarray
    idx: np.ndarray
    groups: np.ndarray
    fold_id: np.ndarray
    y_pred_f1: np.ndarray
    y_pred_mcc: np.ndarray
    y_pred_youden: np.ndarray
    thresholds_per_fold: List[Dict[str, float]]
    shap_values: Optional[np.ndarray]        # pooled (n, p) over outer-test subsamples, or None
    shap_row_index: Optional[np.ndarray]     # index labels aligned to shap_values rows, or None
    feature_names: List[str]
    fold_diagnostics: pd.DataFrame

def run_nested_cv_once_glm(
    X: pd.DataFrame,
    y: pd.Series,
    groups: pd.Series,
    outer_splits: int,
    inner_splits_threshold: int,
    seed: int,
    run_shap: bool,
    max_shap_samples_per_fold: Optional[int],
    shap_background_n: int,
) -> NestedCVResult:
    y_np = as_numpy(y)
    groups_np = as_numpy(groups)
    feature_names = list(X.columns.astype(str))

    if not (len(X) == len(y_np) == len(groups_np)):
        raise ValueError("X, y, and groups must have identical lengths.")
    if pd.isna(groups_np).any():
        raise ValueError(f"{GROUP_COL} contains missing values.")

    outer_cv = StratifiedGroupKFold(
        n_splits=outer_splits, shuffle=True, random_state=seed
    )

    y_true_all, p_all, idx_all, groups_all, fold_id_all = [], [], [], [], []
    pred_f1_all, pred_mcc_all, pred_youden_all = [], [], []
    thresholds_per_fold: List[Dict[str, float]] = []
    fold_diagnostics = []

    shap_blocks = []
    shap_idx_blocks = []

    for fold, (tr_idx, te_idx) in enumerate(
        outer_cv.split(X, y_np, groups=groups_np), start=1
    ):
        X_tr, y_tr = X.iloc[tr_idx], y_np[tr_idx]
        X_te, y_te = X.iloc[te_idx], y_np[te_idx]
        groups_tr = groups_np[tr_idx]
        groups_te = groups_np[te_idx]

        train_patients = set(groups_tr.tolist())
        validation_patients = set(groups_te.tolist())
        overlap = train_patients.intersection(validation_patients)
        if overlap:
            raise RuntimeError(
                f"Patient leakage detected in outer fold {fold}: "
                f"{len(overlap)} patient(s) occur in both train and validation."
            )

        fold_diagnostics.append({
            "seed": seed,
            "fold": fold,
            "n_train_admissions": len(tr_idx),
            "n_validation_admissions": len(te_idx),
            "n_train_patients": len(train_patients),
            "n_validation_patients": len(validation_patients),
            "n_overlapping_patients": len(overlap),
        })

        # Leakage-free threshold selection on the outer-training data using
        # patient-grouped OOF predictions.
        base_model = make_glm_pipeline(random_state=seed)
        p_tr_oof = get_oof_probabilities(
            model=base_model,
            X=X_tr,
            y=y_tr,
            groups=groups_tr,
            n_splits=inner_splits_threshold,
            seed=seed,
        )
        thr = thresholds_from_predictions(y_tr, p_tr_oof)
        thresholds_per_fold.append(thr)

        # Fit on the complete outer-training partition and predict outer-test.
        model = make_glm_pipeline(random_state=seed)
        model.fit(X_tr, y_tr)
        p_te = model.predict_proba(X_te)[:, 1]

        y_pred_f1     = (p_te >= thr["F1-optimal"]).astype(int)
        y_pred_mcc    = (p_te >= thr["MCC-optimal"]).astype(int)
        y_pred_youden = (p_te >= thr["Youden"]).astype(int)

        y_true_all.append(y_te)
        p_all.append(p_te)
        idx_all.append(X_te.index.to_numpy())
        groups_all.append(groups_te)
        fold_id_all.append(np.full(len(te_idx), fold, dtype=int))

        pred_f1_all.append(y_pred_f1)
        pred_mcc_all.append(y_pred_mcc)
        pred_youden_all.append(y_pred_youden)

        # SHAP on this outer-test fold. Because the outer split is patient-grouped,
        # the SHAP background and explained rows are also patient-disjoint.
        if run_shap:
            sv, idx_labels = shap_for_glm_pipeline(
                fitted_model=model,
                X_train_fold=X_tr,
                X_test_fold=X_te,
                feature_names=feature_names,
                max_shap_samples=max_shap_samples_per_fold,
                background_n=shap_background_n,
                seed=seed + fold,
            )
            shap_blocks.append(sv)
            shap_idx_blocks.append(idx_labels)

    shap_values = None
    shap_row_index = None
    if run_shap and len(shap_blocks) > 0:
        shap_values = np.vstack(shap_blocks)
        shap_row_index = np.concatenate(shap_idx_blocks)

    return NestedCVResult(
        seed=seed,
        y_true=np.concatenate(y_true_all),
        p=np.concatenate(p_all),
        idx=np.concatenate(idx_all),
        groups=np.concatenate(groups_all),
        fold_id=np.concatenate(fold_id_all),
        y_pred_f1=np.concatenate(pred_f1_all),
        y_pred_mcc=np.concatenate(pred_mcc_all),
        y_pred_youden=np.concatenate(pred_youden_all),
        thresholds_per_fold=thresholds_per_fold,
        shap_values=shap_values,
        shap_row_index=shap_row_index,
        feature_names=feature_names,
        fold_diagnostics=pd.DataFrame(fold_diagnostics),
    )


# ============================
# Patient/admission summaries + group alignment
# ============================

def summarize_admissions_per_patient(
    df: pd.DataFrame,
    cohort_name: str,
    group_col: str = GROUP_COL,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Summarize unique patients and the distribution of admissions per patient."""
    if group_col not in df.columns:
        raise KeyError(f"{group_col!r} is not present in cohort {cohort_name!r}.")
    if df[group_col].isna().any():
        raise ValueError(f"{cohort_name}: {group_col} contains missing values.")

    counts = df.groupby(group_col, dropna=False).size().rename("n_admissions")
    q = counts.quantile([0.25, 0.50, 0.75])
    summary = pd.DataFrame([{
        "cohort": cohort_name,
        "n_admissions": int(len(df)),
        "n_unique_patients": int(counts.size),
        "n_patients_with_recurrent_admissions": int((counts > 1).sum()),
        "pct_patients_with_recurrent_admissions": float(100 * (counts > 1).mean()),
        "recurrent_admissions_present": bool((counts > 1).any()),
        "admissions_per_patient_mean": float(counts.mean()),
        "admissions_per_patient_sd": float(counts.std(ddof=1)) if counts.size > 1 else 0.0,
        "admissions_per_patient_min": int(counts.min()),
        "admissions_per_patient_q1": float(q.loc[0.25]),
        "admissions_per_patient_median": float(q.loc[0.50]),
        "admissions_per_patient_q3": float(q.loc[0.75]),
        "admissions_per_patient_max": int(counts.max()),
    }])

    distribution = (
        counts.value_counts().sort_index().rename_axis("n_admissions")
        .rename("n_patients").reset_index()
    )
    distribution.insert(0, "cohort", cohort_name)
    return summary, distribution

def align_groups_to_model_rows(
    source_df: pd.DataFrame,
    X: pd.DataFrame,
    group_col: str = GROUP_COL,
) -> pd.Series:
    """Align patient identifiers from the source cohort to rows returned by preprocessing."""
    if group_col not in source_df.columns:
        raise KeyError(f"{group_col!r} not found in the source training dataframe.")
    if source_df[group_col].isna().any():
        raise ValueError(f"{group_col} contains missing values.")

    # Preferred case: preprocessing preserved the source row index.
    if X.index.equals(source_df.index):
        groups = source_df.loc[X.index, group_col].copy()
        groups.index = X.index
        return groups

    ambiguous_reset_index = (
        len(X) < len(source_df)
        and isinstance(X.index, pd.RangeIndex)
        and X.index.start == 0
        and X.index.step == 1
    )
    if (
        not ambiguous_reset_index
        and source_df.index.is_unique
        and X.index.isin(source_df.index).all()
    ):
        groups = source_df.loc[X.index, group_col].copy()
        groups.index = X.index
        return groups

    # Safe only when no rows were removed/reordered.
    if len(source_df) == len(X):
        warnings.warn(
            "X_train does not retain a directly mappable source index. "
            "Assuming preprocessing preserved row order when aligning subject_reference. "
            "For maximum safety, preserve the original DataFrame index in do_train_test_split()."
        )
        return pd.Series(source_df[group_col].to_numpy(), index=X.index, name=group_col)

    raise ValueError(
        "Could not safely align subject_reference to X_train. "
        "Preserve the original row index through do_train_test_split(), "
        "or return patient identifiers alongside X/y."
    )

# Reviewer-requested cohort-level admission summaries.
cohort_summary_frames = []
cohort_distribution_frames = []
for _cohort_name, _df in [
    ("internal", data_fs_static_train),
    ("external", data_fs_static_external),
]:
    if GROUP_COL in _df.columns:
        _summary, _distribution = summarize_admissions_per_patient(_df, _cohort_name)
        cohort_summary_frames.append(_summary)
        cohort_distribution_frames.append(_distribution)
    else:
        print(f"Skipping patient/admission summary for {_cohort_name}: {GROUP_COL!r} not found.")

if cohort_summary_frames:
    cohort_patient_summary_df = pd.concat(cohort_summary_frames, ignore_index=True)
    cohort_patient_summary_path = os.path.join(OUT_DIR, "cohort_patient_admission_summary.csv")
    cohort_patient_summary_df.to_csv(cohort_patient_summary_path, index=False)
    print(f"Saved cohort patient/admission summary: {cohort_patient_summary_path}")
    print(cohort_patient_summary_df)

if cohort_distribution_frames:
    admissions_per_patient_distribution_df = pd.concat(
        cohort_distribution_frames, ignore_index=True
    )
    admissions_per_patient_distribution_path = os.path.join(
        OUT_DIR, "admissions_per_patient_distribution.csv"
    )
    admissions_per_patient_distribution_df.to_csv(
        admissions_per_patient_distribution_path, index=False
    )
    print(
        f"Saved admissions-per-patient distribution: "
        f"{admissions_per_patient_distribution_path}"
    )


# ============================
# Run repeated patient-grouped nested CV
# ============================

if not isinstance(y_train, pd.Series):
    y_train = pd.Series(y_train, index=X_train.index)

X = X_train.copy()
y = y_train.copy()
groups = align_groups_to_model_rows(data_fs_static_train, X, GROUP_COL)

if GROUP_COL in X.columns:
    raise ValueError(
        f"{GROUP_COL} is present in X. Remove patient identifiers from feature_space before modeling."
    )

if not (X.index.equals(y.index) and X.index.equals(groups.index)):
    y = y.reindex(X.index)
    groups = groups.reindex(X.index)

if y.isna().any() or groups.isna().any():
    raise ValueError("Could not align X, y, and subject_reference without missing values.")

print(
    f"Internal modeling cohort: {len(X)} admissions from "
    f"{groups.nunique()} unique patients; "
    f"{(groups.value_counts() > 1).sum()} patients have recurrent admissions."
)

all_trial_preds: List[NestedCVResult] = []

print(f"Running repeated patient-grouped nested CV (GLM) ({N_NESTED_TRIALS} trials)...")
for i, seed in enumerate(RANDOM_SEEDS, start=1):
    print(f"\n=== Trial {i}/{len(RANDOM_SEEDS)} | seed={seed} ===")
    res = run_nested_cv_once_glm(
        X=X,
        y=y,
        groups=groups,
        outer_splits=OUTER_SPLITS,
        inner_splits_threshold=INNER_SPLITS_THRESHOLD,
        seed=seed,
        run_shap=RUN_SHAP,
        max_shap_samples_per_fold=MAX_SHAP_SAMPLES_PER_FOLD,
        shap_background_n=SHAP_BACKGROUND_N,
    )
    all_trial_preds.append(res)

    # Additional global diagnostic: within a trial, each patient must occur in
    # exactly one outer validation fold.
    patient_fold_counts = (
        pd.DataFrame({"patient": res.groups, "fold": res.fold_id})
        .groupby("patient")["fold"].nunique()
    )
    if (patient_fold_counts > 1).any():
        raise RuntimeError(
            "A patient appears in more than one outer validation fold; grouped CV failed."
        )

print("\nNested CV runs complete.")

# Save explicit reviewer-facing fold diagnostics.
fold_diagnostics_df = pd.concat(
    [r.fold_diagnostics for r in all_trial_preds], ignore_index=True
)
fold_diagnostics_path = os.path.join(OUT_DIR, "grouped_cv_fold_diagnostics.csv")
fold_diagnostics_df.to_csv(fold_diagnostics_path, index=False)
print(f"Saved grouped CV diagnostics: {fold_diagnostics_path}")
print(
    "Maximum patient overlap across outer folds:",
    int(fold_diagnostics_df["n_overlapping_patients"].max()),
)


# ============================
# Metrics + bootstrapped distributional CIs pooled across trials
# ============================

METRICS_TO_REPORT = [
    "accuracy","precision","recall","specificity","sensitivity","npv",
    "f1","mcc","auc","auprc","brier"
]
RULES = ["F1-optimal", "MCC-optimal", "Youden"]

# Per-trial point estimates (pooled outer-test predictions within trial)
trial_point_rows = []
for t, res in enumerate(all_trial_preds, start=1):
    rule_to_pred = {
        "F1-optimal": res.y_pred_f1,
        "MCC-optimal": res.y_pred_mcc,
        "Youden": res.y_pred_youden,
    }
    for rule, y_pred in rule_to_pred.items():
        m = point_metrics(res.y_true, res.p, y_pred)
        trial_point_rows.append({"trial": t, "seed": res.seed, "rule": rule,
                                 **{k: m[k] for k in METRICS_TO_REPORT}})

trial_point_df = pd.DataFrame(trial_point_rows)
trial_point_path = os.path.join(OUT_DIR, "nested_glm_trial_point_metrics.csv")
trial_point_df.to_csv(trial_point_path, index=False)
print(f"Saved per-trial point estimates: {trial_point_path}")

# Bootstrap distributions within each trial; pool across trials
all_boot_rows = []
for t, res in enumerate(all_trial_preds, start=1):
    rule_to_pred = {
        "F1-optimal": res.y_pred_f1,
        "MCC-optimal": res.y_pred_mcc,
        "Youden": res.y_pred_youden,
    }
    for rule, y_pred in rule_to_pred.items():
        boot_seed = int(res.seed + (hash(rule) % 10_000))
        dist = bootstrap_metric_distribution(
            y_true=res.y_true,
            y_prob=res.p,
            y_pred=y_pred,
            groups=res.groups,
            metrics=METRICS_TO_REPORT,
            n_boot=N_BOOTSTRAPS,
            seed=boot_seed,
        )
        dist["trial"] = t
        dist["seed"] = res.seed
        dist["rule"] = rule
        all_boot_rows.append(dist)

boot_df = pd.concat(all_boot_rows, ignore_index=True)
boot_path = os.path.join(OUT_DIR, f"nested_glm_cluster_bootstrap_distributions_{N_BOOTSTRAPS}x{N_NESTED_TRIALS}.csv")
boot_df.to_csv(boot_path, index=False)
print(f"Saved bootstrap distributions (may be large): {boot_path}")

# Summarize pooled bootstrap distributions per rule+metric
summary_rows = []
for rule in RULES:
    for metric in METRICS_TO_REPORT:
        g = boot_df[(boot_df["rule"] == rule) & (boot_df["metric"] == metric)]["boot_value"].to_numpy(dtype=float)

        ci_low = float(np.nanpercentile(g, 100 * (BOOT_ALPHA / 2)))
        ci_high = float(np.nanpercentile(g, 100 * (1 - BOOT_ALPHA / 2)))
        pe = float(trial_point_df[trial_point_df["rule"] == rule][metric].mean())

        summary_rows.append({
            "rule": rule,
            "metric": metric,
            "point_estimate_mean_over_trials": pe,
            "bootstrap_ci_low": ci_low,
            "bootstrap_ci_high": ci_high,
            "n_boot_total": int(len(g)),
        })

bootstrap_summary_df = pd.DataFrame(summary_rows)
bootstrap_summary_path = os.path.join(OUT_DIR, "nested_glm_cluster_bootstrap_CI_summary.csv")
bootstrap_summary_df.to_csv(bootstrap_summary_path, index=False)
print(f"Saved bootstrap CI summary: {bootstrap_summary_path}")
print(bootstrap_summary_df)


# ============================
# Calibration: per-trial points + bootstrap distributions pooled across trials
# ============================

cal_point_rows = []
cal_boot_rows = []

for t, res in enumerate(all_trial_preds, start=1):
    cal_point_rows.append({
        "trial": t,
        "seed": res.seed,
        "brier": float(brier_score_loss(res.y_true, res.p)),
        "citl": float(calibration_in_the_large(res.y_true, res.p)),
        "slope": float(calibration_slope(res.y_true, res.p)),
    })

    dist = bootstrap_calibration_distribution(
        y_true=res.y_true,
        y_prob=res.p,
        groups=res.groups,
        n_boot=N_BOOTSTRAPS,
        seed=res.seed + 99_999,
    )
    dist["trial"] = t
    dist["seed"] = res.seed
    cal_boot_rows.append(dist)

cal_point_df = pd.DataFrame(cal_point_rows)
cal_point_path = os.path.join(OUT_DIR, "nested_glm_calibration_trial_points.csv")
cal_point_df.to_csv(cal_point_path, index=False)

cal_boot_df = pd.concat(cal_boot_rows, ignore_index=True)
cal_boot_path = os.path.join(OUT_DIR, f"nested_glm_calibration_cluster_bootstrap_{N_BOOTSTRAPS}x{N_NESTED_TRIALS}.csv")
cal_boot_df.to_csv(cal_boot_path, index=False)

cal_summary_rows = []
for metric in ["brier", "citl", "slope"]:
    g = cal_boot_df[cal_boot_df["metric"] == metric]["boot_value"].to_numpy(dtype=float)
    ci_low = float(np.nanpercentile(g, 100 * (BOOT_ALPHA / 2)))
    ci_high = float(np.nanpercentile(g, 100 * (1 - BOOT_ALPHA / 2)))
    pe = float(cal_point_df[metric].mean())
    cal_summary_rows.append({
        "metric": metric,
        "point_estimate_mean_over_trials": pe,
        "bootstrap_ci_low": ci_low,
        "bootstrap_ci_high": ci_high,
        "n_boot_total": int(len(g)),
    })

cal_summary_df = pd.DataFrame(cal_summary_rows)
cal_summary_path = os.path.join(OUT_DIR, "nested_glm_calibration_cluster_bootstrap_CI_summary.csv")
cal_summary_df.to_csv(cal_summary_path, index=False)

print(f"Saved calibration point estimates: {cal_point_path}")
print(f"Saved calibration bootstrap distributions: {cal_boot_path}")
print(f"Saved calibration bootstrap CI summary: {cal_summary_path}")
print(cal_summary_df)


# ============================
# SHAP aggregation across trials (mean(|SHAP|) over pooled outer-test samples)
# ============================

if RUN_SHAP:
    shap_blocks = [r.shap_values for r in all_trial_preds if r.shap_values is not None]
    if len(shap_blocks) == 0:
        print("\nNo SHAP values were collected (RUN_SHAP=True but results empty).")
    else:
        shap_stack = np.vstack(shap_blocks)  # (sum_n_subsampled, n_features)
        feature_names = all_trial_preds[0].feature_names

        shap_mean = shap_stack.mean(axis=0)
        shap_mean_abs = np.abs(shap_stack).mean(axis=0)

        shap_summary_df = pd.DataFrame({
            "feature": feature_names,
            "mean_shap": shap_mean,
            "mean_abs_shap": shap_mean_abs,
        }).sort_values("mean_abs_shap", ascending=False)

        shap_path = os.path.join(SHAP_DIR, "nested_glm_shap_aggregated_over_20_trials.csv")
        shap_summary_df.to_csv(shap_path, index=False)

        print(f"\nSaved aggregated SHAP: {shap_path}")
        print(shap_summary_df.head(25))

print("\nDone.")
